# USGS Continuous Water-Quality Data Cleaning

This notebook is intentionally restartable.

It loads the saved download plan and the raw per-station CSV files from disk, then:

1. Cleans timestamps and numeric values.
2. Prefers approved observations when available.
3. Resolves duplicate timestamps using the latest `last_modified` record.
4. Converts high-frequency observations to hourly medians.
5. Measures completeness against the planned download window.
6. Saves one cleaned hourly file per station-variable.
7. Builds one merged hourly file per station.
8. Builds one combined all-stations hourly dataset.

The three known no-data series are handled automatically instead of causing the notebook to fail.

In [1]:
import os
import re
import pandas as pd
import numpy as np
from pandas.errors import EmptyDataError

PLAN_FILE = "usgs_download_plan.csv"
RAW_DIR = "usgs_continuous_download"

HOURLY_DIR = "usgs_hourly_clean"
STATION_DIR = "usgs_station_hourly"

AUDIT_FILE = "usgs_cleaning_audit.csv"
COMBINED_FILE = "usgs_all_stations_hourly.csv"

os.makedirs(HOURLY_DIR, exist_ok=True)
os.makedirs(STATION_DIR, exist_ok=True)

print("Working directory:", os.getcwd())
print("Plan exists:", os.path.exists(PLAN_FILE))
print("Raw directory exists:", os.path.isdir(RAW_DIR))

Working directory: /Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph
Plan exists: True
Raw directory exists: True


## 1. Load the saved download plan

In [2]:
download_plan_df = pd.read_csv(PLAN_FILE)

download_plan_df["station_id"] = (
    download_plan_df["station_id"]
    .astype(str)
    .str.strip()
)

download_plan_df["parameter_code"] = (
    download_plan_df["parameter_code"]
    .astype(str)
    .str.replace(".0", "", regex=False)
    .str.zfill(5)
)

download_plan_df["start"] = pd.to_datetime(
    download_plan_df["start"],
    errors="coerce",
    utc=True
)

download_plan_df["end"] = pd.to_datetime(
    download_plan_df["end"],
    errors="coerce",
    utc=True
)

print("Plan rows:", len(download_plan_df))
print("Stations:", download_plan_df["station_id"].nunique())
print()
print(download_plan_df["variable"].value_counts())

Plan rows: 139
Stations: 28

variable
Water temperature       28
Specific conductance    28
Discharge               23
pH                      21
DO concentration        19
Turbidity               19
Stage                    1
Name: count, dtype: int64


## 2. Helper functions

In [3]:
def safe_variable_name(variable):
    return (
        str(variable)
        .lower()
        .replace(" ", "_")
        .replace("/", "_")
    )


def raw_file_path(site_id, variable):
    return os.path.join(
        RAW_DIR,
        str(site_id),
        f"{safe_variable_name(variable)}.csv"
    )


def hourly_file_path(site_id, variable):
    station_dir = os.path.join(
        HOURLY_DIR,
        str(site_id)
    )
    os.makedirs(station_dir, exist_ok=True)

    return os.path.join(
        station_dir,
        f"{safe_variable_name(variable)}.csv"
    )


def clean_raw_series(file_path, variable):
    try:
        df = pd.read_csv(
            file_path,
            low_memory=False
        )
    except EmptyDataError:
        return None, {
            "status": "empty_file",
            "raw_rows": 0,
            "clean_rows": 0
        }
    except Exception as e:
        return None, {
            "status": f"read_error: {e}",
            "raw_rows": -1,
            "clean_rows": 0
        }

    raw_rows = len(df)

    if raw_rows == 0:
        return None, {
            "status": "zero_rows",
            "raw_rows": 0,
            "clean_rows": 0
        }

    required = {"time", "value"}

    if not required.issubset(df.columns):
        return None, {
            "status": "missing_time_or_value",
            "raw_rows": raw_rows,
            "clean_rows": 0
        }

    df = df.copy()

    df["time"] = pd.to_datetime(
        df["time"],
        errors="coerce",
        utc=True
    )

    df["value"] = pd.to_numeric(
        df["value"],
        errors="coerce"
    )

    df = df.dropna(
        subset=["time", "value"]
    )

    # If approved observations exist, prefer them.
    if "approval_status" in df.columns:
        approved_mask = (
            df["approval_status"]
            .fillna("")
            .astype(str)
            .str.strip()
            .str.lower()
            .eq("approved")
        )

        if approved_mask.any():
            df = df[approved_mask].copy()

    # If multiple records occur at the same timestamp,
    # keep the most recently modified record.
    if "last_modified" in df.columns:
        df["last_modified"] = pd.to_datetime(
            df["last_modified"],
            errors="coerce",
            utc=True
        )

        df = df.sort_values(
            ["time", "last_modified"]
        )
    else:
        df = df.sort_values("time")

    df = (
        df
        .drop_duplicates(
            subset=["time"],
            keep="last"
        )
        [["time", "value"]]
        .rename(columns={"value": variable})
        .sort_values("time")
        .reset_index(drop=True)
    )

    if df.empty:
        return None, {
            "status": "no_valid_values",
            "raw_rows": raw_rows,
            "clean_rows": 0
        }

    return df, {
        "status": "ok",
        "raw_rows": raw_rows,
        "clean_rows": len(df)
    }


def to_hourly(clean_df, variable, start, end):
    start = pd.Timestamp(start)
    end = pd.Timestamp(end)

    if start.tzinfo is None:
        start = start.tz_localize("UTC")
    else:
        start = start.tz_convert("UTC")

    if end.tzinfo is None:
        end = end.tz_localize("UTC")
    else:
        end = end.tz_convert("UTC")

    # Use the planned time window as the completeness denominator.
    first_hour = start.floor("h")
    last_hour = end.ceil("h")

    full_index = pd.date_range(
        first_hour,
        last_hour,
        freq="1h",
        tz="UTC"
    )

    hourly = (
        clean_df
        .set_index("time")[[variable]]
        .resample("1h")
        .median()
        .reindex(full_index)
    )

    hourly.index.name = "time"
    hourly = hourly.reset_index()

    return hourly

## 3. Clean every downloaded series and convert to hourly medians

This cell safely skips the three known station-variable combinations that returned no data.

In [4]:
audit_rows = []

for i, row in download_plan_df.iterrows():

    site_id = str(row["station_id"])
    variable = row["variable"]
    start = row["start"]
    end = row["end"]

    file_path = raw_file_path(
        site_id,
        variable
    )

    record = {
        "station_id": site_id,
        "variable": variable,
        "parameter_code": row["parameter_code"],
        "planned_start": start,
        "planned_end": end,
        "raw_file": file_path
    }

    if not os.path.exists(file_path):
        record.update({
            "status": "missing_file",
            "raw_rows": -1,
            "clean_rows": 0,
            "hourly_nonnull": 0,
            "expected_hours": 0,
            "completeness_pct": 0.0,
            "first_valid_time": pd.NaT,
            "last_valid_time": pd.NaT
        })

        audit_rows.append(record)
        continue

    clean_df, info = clean_raw_series(
        file_path,
        variable
    )

    record.update(info)

    if clean_df is None:
        record.update({
            "hourly_nonnull": 0,
            "expected_hours": 0,
            "completeness_pct": 0.0,
            "first_valid_time": pd.NaT,
            "last_valid_time": pd.NaT
        })

        audit_rows.append(record)
        print(
            f"[{i+1}/{len(download_plan_df)}] "
            f"{site_id} | {variable} | {record['status']}"
        )
        continue

    hourly = to_hourly(
        clean_df,
        variable,
        start,
        end
    )

    out_file = hourly_file_path(
        site_id,
        variable
    )

    temp_file = out_file + ".tmp"

    hourly.to_csv(
        temp_file,
        index=False
    )

    os.replace(
        temp_file,
        out_file
    )

    hourly_nonnull = int(
        hourly[variable].notna().sum()
    )

    expected_hours = len(hourly)

    completeness_pct = (
        100.0 * hourly_nonnull / expected_hours
        if expected_hours > 0
        else 0.0
    )

    valid_times = hourly.loc[
        hourly[variable].notna(),
        "time"
    ]

    record.update({
        "status": "ok",
        "hourly_nonnull": hourly_nonnull,
        "expected_hours": expected_hours,
        "completeness_pct": completeness_pct,
        "first_valid_time": (
            valid_times.min()
            if len(valid_times)
            else pd.NaT
        ),
        "last_valid_time": (
            valid_times.max()
            if len(valid_times)
            else pd.NaT
        )
    })

    audit_rows.append(record)

    print(
        f"[{i+1}/{len(download_plan_df)}] "
        f"{site_id} | {variable} | "
        f"{hourly_nonnull:,}/{expected_hours:,} hours | "
        f"{completeness_pct:.1f}%"
    )


cleaning_audit = pd.DataFrame(
    audit_rows
)

cleaning_audit.to_csv(
    AUDIT_FILE,
    index=False
)

print()
print("Saved:", AUDIT_FILE)

[1/139] USGS-09070500 | DO concentration | 4,140/8,761 hours | 47.3%
[2/139] USGS-09070500 | Water temperature | 4,140/8,761 hours | 47.3%
[3/139] USGS-09070500 | Specific conductance | 4,113/8,761 hours | 46.9%
[4/139] USGS-09070500 | Turbidity | 3,180/8,761 hours | 36.3%
[5/139] USGS-09070500 | pH | 4,140/8,761 hours | 47.3%
[6/139] USGS-09070500 | Discharge | 7,480/8,761 hours | 85.4%
[7/139] USGS-09071750 | DO concentration | 7,363/8,762 hours | 84.0%
[8/139] USGS-09071750 | Water temperature | 7,375/8,762 hours | 84.2%
[9/139] USGS-09071750 | Specific conductance | 7,375/8,762 hours | 84.2%
[10/139] USGS-09071750 | Turbidity | 4,714/8,762 hours | 53.8%
[11/139] USGS-09071750 | pH | 7,375/8,762 hours | 84.2%
[12/139] USGS-09085000 | DO concentration | 7,316/8,762 hours | 83.5%
[13/139] USGS-09085000 | Water temperature | 7,383/8,762 hours | 84.3%
[14/139] USGS-09085000 | Specific conductance | 7,383/8,762 hours | 84.3%
[15/139] USGS-09085000 | Turbidity | 5,381/8,762 hours | 61.4%


## 4. Review cleaning results

In [5]:
print("Series in plan:", len(cleaning_audit))
print()

print("Status counts:")
print(
    cleaning_audit["status"]
    .value_counts(dropna=False)
)

print()

successful = cleaning_audit[
    cleaning_audit["status"] == "ok"
].copy()

print("Successful series:", len(successful))

if len(successful):
    print()
    print("Completeness summary:")
    print(
        successful["completeness_pct"]
        .describe()
    )

Series in plan: 139

Status counts:
status
ok            136
empty_file      3
Name: count, dtype: int64

Successful series: 136

Completeness summary:
count    136.000000
mean      69.469181
std       25.938776
min        1.438192
25%       59.135412
50%       68.848436
75%       88.694213
max      100.000000
Name: completeness_pct, dtype: float64


In [6]:
cleaning_audit[
    [
        "station_id",
        "variable",
        "status",
        "raw_rows",
        "clean_rows",
        "hourly_nonnull",
        "expected_hours",
        "completeness_pct",
        "first_valid_time",
        "last_valid_time"
    ]
].sort_values(
    ["status", "completeness_pct", "station_id", "variable"]
).head(40)

,station_id,variable,status,raw_rows,clean_rows,hourly_nonnull,expected_hours,completeness_pct,first_valid_time,last_valid_time
98,USGS-09365000,Discharge,empty_file,0,0,0,0,0.000000,NaT,NaT
122,USGS-09404200,Specific conductance,empty_file,0,0,0,0,0.000000,NaT,NaT
121,USGS-09404200,Water temperature,empty_file,0,0,0,0,0.000000,NaT,NaT
106,USGS-09371010,Turbidity,ok,4100,489,126,8761,1.438192,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
101,USGS-09368000,Turbidity,ok,4045,524,134,8761,1.529506,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
100,USGS-09368000,Specific conductance,ok,4192,671,168,8761,1.917589,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
102,USGS-09368000,pH,ok,4192,671,168,8761,1.917589,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
105,USGS-09371010,Specific conductance,ok,4285,672,168,8761,1.917589,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
107,USGS-09371010,pH,ok,4285,672,168,8761,1.917589,2023-09-11 00:00:00+00:00,2023-09-17 23:00:00+00:00
104,USGS-09371010,Water temperature,ok,6465,2180,550,8761,6.277822,2022-11-15 15:00:00+00:00,2022-12-09 06:00:00+00:00


## 5. Build one merged hourly dataset per station

Each station gets one row per hour over its planned window and one column per available variable.

In [7]:
station_summaries = []

for site_id in sorted(
    download_plan_df["station_id"].unique()
):

    station_plan = download_plan_df[
        download_plan_df["station_id"] == site_id
    ].copy()

    station_frames = []

    for _, row in station_plan.iterrows():
        variable = row["variable"]

        file_path = hourly_file_path(
            site_id,
            variable
        )

        if not os.path.exists(file_path):
            continue

        try:
            df = pd.read_csv(
                file_path,
                parse_dates=["time"]
            )
        except Exception:
            continue

        if variable not in df.columns:
            continue

        df["time"] = pd.to_datetime(
            df["time"],
            errors="coerce",
            utc=True
        )

        station_frames.append(
            df[["time", variable]]
        )

    if not station_frames:
        print(
            "No cleaned series for:",
            site_id
        )
        continue

    merged = station_frames[0]

    for frame in station_frames[1:]:
        merged = merged.merge(
            frame,
            on="time",
            how="outer"
        )

    merged = (
        merged
        .sort_values("time")
        .drop_duplicates("time")
        .reset_index(drop=True)
    )

    merged.insert(
        0,
        "station_id",
        site_id
    )

    station_file = os.path.join(
        STATION_DIR,
        f"{site_id}.csv"
    )

    temp_file = station_file + ".tmp"

    merged.to_csv(
        temp_file,
        index=False
    )

    os.replace(
        temp_file,
        station_file
    )

    data_columns = [
        c
        for c in merged.columns
        if c not in ["station_id", "time"]
    ]

    station_summaries.append({
        "station_id": site_id,
        "rows": len(merged),
        "variables": len(data_columns),
        "variable_names": ", ".join(data_columns)
    })

    print(
        site_id,
        "| rows:",
        len(merged),
        "| variables:",
        len(data_columns)
    )


station_summary = pd.DataFrame(
    station_summaries
)

station_summary.to_csv(
    "usgs_station_hourly_summary.csv",
    index=False
)

USGS-09070500 | rows: 8761 | variables: 6
USGS-09071750 | rows: 8762 | variables: 5
USGS-09085000 | rows: 8762 | variables: 6
USGS-09085150 | rows: 8762 | variables: 5
USGS-09085160 | rows: 8762 | variables: 5
USGS-09092570 | rows: 8762 | variables: 6
USGS-09095500 | rows: 8762 | variables: 6
USGS-09106485 | rows: 1886 | variables: 4
USGS-09144250 | rows: 8762 | variables: 6
USGS-09152500 | rows: 8762 | variables: 6
USGS-09163500 | rows: 8762 | variables: 6
USGS-09180000 | rows: 8762 | variables: 3
USGS-09180500 | rows: 8761 | variables: 6
USGS-09217000 | rows: 8762 | variables: 3
USGS-09234500 | rows: 8762 | variables: 3
USGS-09261000 | rows: 8762 | variables: 6
USGS-09306500 | rows: 4406 | variables: 3
USGS-09315000 | rows: 8762 | variables: 6
USGS-09328960 | rows: 8762 | variables: 3
USGS-09365000 | rows: 8761 | variables: 4
USGS-09368000 | rows: 8761 | variables: 5
USGS-09371010 | rows: 8761 | variables: 5
USGS-09379500 | rows: 8761 | variables: 6
USGS-09380000 | rows: 8761 | varia

## 6. Build one combined all-stations hourly dataset

In [8]:
all_station_frames = []

for site_id in sorted(
    download_plan_df["station_id"].unique()
):

    station_file = os.path.join(
        STATION_DIR,
        f"{site_id}.csv"
    )

    if not os.path.exists(station_file):
        continue

    df = pd.read_csv(
        station_file,
        low_memory=False
    )

    all_station_frames.append(df)


if all_station_frames:

    all_hourly = pd.concat(
        all_station_frames,
        ignore_index=True,
        sort=False
    )

    all_hourly["time"] = pd.to_datetime(
        all_hourly["time"],
        errors="coerce",
        utc=True
    )

    all_hourly = (
        all_hourly
        .sort_values(
            ["station_id", "time"]
        )
        .reset_index(drop=True)
    )

    all_hourly.to_csv(
        COMBINED_FILE,
        index=False
    )

    print(
        "Saved:",
        COMBINED_FILE
    )

    print(
        "Rows:",
        len(all_hourly)
    )

    print(
        "Stations:",
        all_hourly[
            "station_id"
        ].nunique()
    )

    print(
        "Columns:",
        all_hourly.columns.tolist()
    )

else:
    print(
        "No station-level files were found."
    )

Saved: usgs_all_stations_hourly.csv
Rows: 234095
Stations: 28
Columns: ['station_id', 'time', 'DO concentration', 'Water temperature', 'Specific conductance', 'Turbidity', 'pH', 'Discharge', 'Stage']


## 7. Station-variable completeness table

This makes it easy to decide which series are good enough for modeling.

In [9]:
completeness_table = (
    cleaning_audit[
        cleaning_audit["status"] == "ok"
    ]
    .pivot_table(
        index="station_id",
        columns="variable",
        values="completeness_pct",
        aggfunc="first"
    )
    .round(1)
)

completeness_table.to_csv(
    "usgs_station_variable_completeness.csv"
)

completeness_table

variable,DO concentration,Discharge,Specific conductance,Stage,Turbidity,Water temperature,pH
station_id,,,,,,,
USGS-09070500,47.3,85.4,46.9,NaN,36.3,47.3,47.3
USGS-09071750,84.0,NaN,84.2,NaN,53.8,84.2,84.2
USGS-09085000,83.5,13.3,84.3,NaN,61.4,84.3,84.3
USGS-09085150,99.9,NaN,99.9,NaN,91.2,99.9,99.9
USGS-09085160,84.2,NaN,83.3,NaN,61.1,84.4,84.4
USGS-09092570,58.6,57.1,58.5,NaN,49.0,58.6,58.5
USGS-09095500,61.2,47.0,85.0,NaN,58.0,85.8,61.2
USGS-09106485,99.9,99.4,99.9,NaN,NaN,99.9,NaN
USGS-09144250,65.3,70.6,65.3,NaN,65.3,65.3,65.3
